# 从推理到行动：工具使用与代码反馈


> 前两讲处理的是静态模型：02 讲把多次采样与投票用在推理阶段，03 讲让模型检查模型的输出。这些方法都不让模型与环境交互，模型知道多少，取决于训练数据里有多少。
>
> 这一讲把模型与环境连成一个闭环：输出动作、执行工具、读回观察。我们按反馈的来源分成三个层次：环境反馈（ReAct）、执行反馈（RLEF）、AI 反馈（Constitutional AI）。


我们每天用 ChatGPT，在对话框里打一句话，它回一段文字。但只靠对话框，它不能真的替我们做外面的事：让它"把这段代码跑一下"，它只能把代码念一遍；让它"把这个文件里的数字改掉"，它碰不到我们的文件。原因很简单：模型的输入和输出都只是文字，它本身不会去调用任何外部程序，也不会读取运行环境。

这一讲要解决的就是"让模型真正做事"的问题。做法是在模型外面套一层程序：模型输出一段文字，说明自己想调用哪个工具；这层程序解析文字，真的去执行工具，比如检索网页、运行代码、读写文件；工具的执行结果再作为文字喂回模型，模型据此决定下一步。有了这层程序，模型就从"只能说话"变成"能做事"。学完这一讲，你能亲手实现这样一个最小循环：定义工具、写一个解析器把模型的文字翻译成工具调用、把执行结果（包括报错）当作反馈喂回模型，让模型根据反馈修正自己的回答。

放在第 1 讲的最小循环里看，行动和反馈这两步当时只做了一种最简单的形态：动作是算术调用，反馈是数字结果。这一讲把这两步做透：工具怎么定义、模型怎么表达调用、跑代码得到的报错或通过怎么作为信号让模型修正自己。内容按反馈的来源分成三块，层层递进：先让 Agent 拿到环境反馈，再让反馈参与训练，最后让 AI 也成为反馈的提供者。我们从最基础的一步开始：模型只输出文字，怎样让这段文字变成一次真实的行动。

## 1. 从推理到行动：ReAct 循环

这一节解决一个问题：模型只思考、不行动，行不行。第 1 讲的最小循环已经让模型输出动作、调用工具，这一节先看"只思考"的方案差在哪里，再介绍把思考与行动交替起来的 ReAct 循环，最后打印一条轨迹认识它的三种片段。

先看只思考的方案。模型在给出答案前，常常先写一段自己的推理，把推理和答案一起输出。这种"先想后答"的做法叫思维链，英文是 Chain-of-Thought，简称 CoT。思维链让模型想得更久，但它只用到模型记住的事实，不接触外部世界。当问题需要模型不知道的知识时，模型编不出真答案，就顺着直觉编一个看起来合理的答案。推理链越长，编造的机会越多，错误还可能沿链往下传。

ReAct 换一个思路：不想完再动，而是想一步、动一步。模型先输出一段想法（Thought），说明自己想干什么；再输出一个动作（Action），指定调用哪个工具；工具执行返回的结果（Observation）作为下一轮思考的依据。三个片段交替出现，就是 ReAct 名字里 Reasoning（推理）与 Acting（行动）各承担的角色。

轨迹里的 Observation 必须来自真实工具执行，是插入进来的文本，而不是模型续写的。在 ALFWorld 的 134 局任务里，ReAct 的最好 6 次取均值达到 71%，纯行动的 Act 只有 45%——思考让行动不迷失方向。下面先打印一条论文风格的轨迹，认识循环的三种片段。

先用一个具体例子看 CoT 的边界。假设问模型《现代文学》创刊于哪一年，而训练数据里没有这本刊物的确切信息。CoT 先写一段推理再给出答案，推理不接触任何外部资料，模型只能从内部记忆里找。找不到时，模型通常不回答"不知道"，而是顺着"文学刊物创刊一般较早"这类直觉编一个年份。推理链越长，编造的机会越多。

同一个问题交给 ReAct，模型不用只靠记忆。模型先写下当前的想法，再发出检索动作，工具返回的观察成为下一步推理的事实来源。思考为行动提供依据，行动的观察又修正下一步的思考。下面把一条 ReAct 轨迹按角色打印出来，我们逐行读一遍。

In [ ]:
# 一条 ReAct 轨迹，按角色逐行打印，认识循环的三种片段
trajectory = [
    ("Thought", "我需要先查到这本杂志的创刊年份。"),
    ("Action", "Search[现代文学]"),
    ("Observation", "《现代文学》是 1923 年创刊于上海的文学刊物。"),
    ("Thought", "创刊年份是 1923 年，晚于 1919 年。"),
    ("Action", "Finish[之后]"),
]

for kind, content in trajectory:
    print(f"{kind:12s}{content}")
print()
print("关键观察：Action 会调用工具，Observation 来自工具执行，")
print("Thought 只更新上下文，不直接产生观察。")


上面打印的轨迹有五条记录，覆盖 ReAct 的三种片段。我们逐条读一遍。

第一条是 Thought 片段：我需要先查到这本杂志的创刊年份。它出现在动作之前，作用是把这个子目标写进上下文。Thought 不调用工具，不产生环境观察，只更新模型自己看到的历史。

第二条是 Action 片段：Search[现代文学]。Action 描述的是调用哪个工具、传什么参数。工具名是 Search，参数是"现代文学"。模型只负责决定动作，不负责执行——动作会交给解析器和工具去运行。

第三条是 Observation 片段：《现代文学》是 1923 年创刊于上海的文学刊物。它来自工具的真实执行，是插入到上下文里的外部文本，而不是模型自己续写的。这是 ReAct 与 CoT 的分界线：Observation 里的每个事实都必须有环境作为来源。

第四条 Thought 片段读取上一条观察里的数字 1923，与题目给出的 1919 做比较，得出"之后"的结论。第五条 Action 片段 Finish[之后] 是终止动作，参数是最终答案，循环看到它就停止。

把五条记录连起来，数据流的形状是固定的：Thought 写计划 → Action 调工具 → Observation 还回结果 → 再 Thought → 再 Action。思考为行动提供依据，行动的观察修正下一步的思考，两种能力互相支撑。

## 2. 工具的定义与调用

这一节回答一个问题：工具是什么，模型怎样表示一次调用。ReAct 循环要转起来，先得有工具可调，还得有办法让模型表达"我想调用哪个工具"。我们先分清循环里两类不同内容，再看论文为维基百科设计的三个动作，最后写一个迷你百科，让循环在没有网络的本地跑起来。

先分清循环里两类内容。模型每轮的输出里，一类是自己想的话，我们叫它 Thought，它不改变环境，只更新模型看到的上下文，更新后的上下文记作 $c_{t+1} = (c_t, \hat{a}_t)$。另一类是真正要执行的工具调用，记作 $a_t$，它在环境里执行，产生观察 $o_{t+1}$。ReAct 的动作空间是这两者的并集 $\hat{A} = A \cup L$，也就是把"写一段想法"也当作动作的一部分。

论文为维基百科设计了三个动作，刻意比真实检索器弱，用来模拟人的检索方式：
- `search[实体]`：返回实体页面的前几句；找不到时给出若干相似实体。
- `lookup[字符串]`：返回页面里包含该字符串的下一个句子，相当于浏览器里的 Ctrl+F。
- `finish[答案]`：结束任务并给出答案。

三个动作覆盖了"找资料、看细节、收尾"三种需求。下面的迷你百科用同一个接口，内置几条本地条目，让循环可以在没有网络的环境里跑起来。

In [ ]:
class MiniWiki:
    """一个迷你百科：内置几条本地条目，模拟论文里的弱化检索接口。

    与论文动作对应：search(实体) 返回页面开头，lookup(关键词) 返回
    当前页里包含关键词的下一个句子。lookup 用游标记录搜索位置。
    """

    def __init__(self):
        self.pages = {
            "现代文学": [
                "《现代文学》是 1923 年创刊于上海的文学刊物。",
                "鲁迅、茅盾等作家曾在该刊物上发表作品。",
                "刊物出版延续到 1930 年代初。",
            ],
            "五四运动": [
                "五四运动发生于 1919 年 5 月 4 日。",
                "运动以北京学生游行开始，随后扩展到全国。",
                "它被视为中国现代史的开端之一。",
            ],
        }
        self.current_page = []
        self.pos = 0

    def search(self, entity):
        """返回实体页面开头两句；找不到时给出相似实体名。"""
        self.current_page = self.pages.get(entity, [])
        self.pos = 0
        if self.current_page:
            return " ".join(self.current_page[:2])
        similar = [name for name in self.pages if entity in name]
        return f"未找到「{entity}」，相似实体：{similar if similar else '无'}"

    def lookup(self, keyword):
        """从游标起返回第一个含关键词的句子，找不到返回提示。"""
        for i in range(self.pos, len(self.current_page)):
            if keyword in self.current_page[i]:
                self.pos = i + 1
                return self.current_page[i]
        self.pos = len(self.current_page)
        return "未找到包含该关键词的句子"


wiki = MiniWiki()
print(wiki.search("现代文学"))
print(wiki.lookup("创刊"))
print(wiki.search("一个不存在的实体"))


MiniWiki 用同一个接口模拟论文里的弱化检索器。两个方法对应论文的两个动作，我们把三次调用逐一对照输出看一遍。

第一次调用 `wiki.search("现代文学")`。search 先做查表：pages.get("现代文学", []) 命中内置条目，把 current_page 设为该条目的三句话列表，pos 重置为 0，返回前两句拼成的一段文本。输出的第一行就是这一页的摘要，相当于搜索结果页的正文开头。

第二次调用 `wiki.lookup("创刊")`。lookup 不在全库找，而是在 current_page 指向的当前页里找，这正是浏览器的 Ctrl+F：先打开页面，再在页面内查找关键词。lookup 从游标 pos 开始逐句扫描，第一句包含"创刊"，于是返回这一句，并把 pos 推进到 1。游标的意义在于连续调用 lookup 时不会反复返回同一句，而是像人工滚动页面一样越找越深。

第三次调用 `wiki.search("一个不存在的实体")`。pages.get 返回空列表，current_page 被清空。search 找不到条目时不报错，而是收集名称里包含该实体的页面名作为相似实体提示；这里一个都没有，返回"未找到「一个不存在的实体」，相似实体：无"。

注意 search 与 lookup 的状态依赖：lookup 依赖 search 先设置好 current_page。如果先调用 lookup，current_page 还是空的，什么也找不到。工具之间需要配合使用，这正是 Agent 循环里"先找页面、再查细节"顺序的来源。

**动作解析器**

模型的输出是自由文本，需要解析成结构化动作才能执行。解析器要宽容：既接受论文格式 `Action: Search[实体]`，也接受函数调用格式 `Action: search("实体")`；一条回复里可能连着写多个动作，我们把它们按顺序排列，逐个执行。终止信号有两种：`Finish[答案]` 动作，或单独的 `Final Answer: ...` 行。

In [ ]:
import re


def parse_actions(text):
    """从模型回复中按顺序取出所有动作指令。

    支持两种格式：Search[实体] 与 search("实体")。
    返回 [(工具名, 参数), ...]，工具名统一转小写。
    """
    pattern = (
        r"Action\s*\d*\s*[:：]\s*([A-Za-z]+)"
        r"\s*(?:\[([^\]]*)\]|\(\s*(?:\"([^\"]*)\"|'([^']*)'|([^)]*))\s*\))"
    )
    actions = []
    for m in re.finditer(pattern, text):
        name = m.group(1).lower()
        arg = (m.group(2) or m.group(3) or m.group(4) or m.group(5) or "").strip()
        actions.append((name, arg))
    return actions


def extract_final_answer(text):
    """提取回复里的最终答案，找不到返回 None。

    识别 Final Answer 标记与 Finish[答案] 动作。
    """
    m = re.search(r"(?:Final Answer|final answer)\s*[:：]\s*([^\n]+)", text)
    if m:
        return m.group(1).strip()
    m = re.search(r"Action\s*\d*\s*[:：]\s*Finish\s*\[([^\]]*)\]", text)
    if m:
        return m.group(1).strip()
    return None


paper_reply = (
    "Thought: 我需要先检索创刊年份。\n"
    "Action 1: Search[现代文学]\n"
    "Thought: 已经拿到年份。\n"
    "Action 2: Finish[之后]"
)
print("解析出的动作：", parse_actions(paper_reply))
print("最终答案：", extract_final_answer(paper_reply))

scripted_reply = (
    'Thought: 脚本化示例 脚本化推理，先搜索再总结。\n'
    'Action: search("CS329A self-improving agents")\n'
    'Final Answer: 真实 API 演示返回占位结论。'
)
print("解析出的动作：", parse_actions(scripted_reply))
print("最终答案：", extract_final_answer(scripted_reply))

assert parse_actions(paper_reply) == [("search", "现代文学"), ("finish", "之后")]
assert extract_final_answer(paper_reply) == "之后"
assert parse_actions(scripted_reply) == [("search", "CS329A self-improving agents")]
assert extract_final_answer(scripted_reply) == "真实 API 演示返回占位结论。"
print("两种格式都能被同一套解析器消化，断言通过。")


解析器的工作是把一段自由文本变成程序可以执行的动作列表。先看输入长什么样，再看正则怎么拆。

模型输出的格式并不统一。论文格式是 `Action: Search[现代文学]`；有的实现让模型写函数调用 `Action: search("现代文学")`；模型还可能写 `Action 1: Search[现代文学]` 这种带编号的写法。三种写法含义相同，解析器要都能读出来。它返回一个列表 [(工具名, 参数), ...]，工具名统一转成小写，方便和 TOOLS 字典的键匹配。

正则分三段理解。第一段 `Action\s*\d*\s*[:：]` 匹配固定前缀：单词 Action、可选空格和编号（\d* 匹配"1"）、冒号（[:：] 同时接受英文冒号和中文冒号）。第二段 `([A-Za-z]+)` 捕获工具名，只认字母，Search、search、Finish 都能匹配。第三段匹配参数，两种写法：方括号里捕获任意非 ] 字符，`Search[现代文学]` 取到"现代文学"；括号里捕获引号包裹的字符串或裸文本，`search("CS329A self-improving agents")` 取到引号里的整段文本。

对一条具体回复手算一遍。`Action 1: Search[现代文学]`：第一段匹配到 `Action 1: `，第二段捕获 Search，第三段从方括号里捕获"现代文学"，得到动作 ("search", "现代文学")。`re.finditer` 按顺序找出所有匹配，所以一条回复里的多个动作按书写顺序返回，循环逐个执行。代码里 m.group(2) or m.group(3) or m.group(4) or m.group(5) 取第一个非空的参数分组：方括号形式只有 group(2) 有值，引号形式只有 group(3) 或 group(4) 有值，最后的 or "" 兜底，参数可以为空。

**循环组装**

循环把工具、解析器和模型组装起来。它维护一段消息历史，每步做四件事：调 LLM 得到一条回复；解析出动作；执行工具；把 Observation 作为一条 user 消息追加进历史，让模型下次能看到。终止条件有两个：出现最终答案，或步数达到上限 max_steps。上限防止死循环——论文点名的一个失败模式是反复生成同一个动作。


In [ ]:
import sys
import os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

client = get_llm()

wiki = MiniWiki()
TOOLS = {"search": wiki.search, "lookup": wiki.lookup}


def run_react(client, question, instructions, tools, max_steps=8):
    """运行一个完整的 ReAct 循环，返回 (最终答案, 轨迹列表)。

    client: llm_client 客户端；question: 用户问题；
    instructions: 写给模型的格式说明；tools: 工具名到函数的映射。
    循环终止于出现最终答案，或超过 max_steps。
    """
    messages = [{"role": "user",
                 "content": instructions + "\n\n问题：" + question}]
    trace = []
    pending = []       # 一条回复里剩余的待执行动作
    finish = None

    for step in range(max_steps):
        if not pending:
            reply = client.chat(messages)
            trace.append("[模型回复]\n" + reply)
            pending = parse_actions(reply)
            finish = extract_final_answer(reply)
            if not pending and finish is not None:
                trace.append("[结束] " + finish)
                return finish, trace
            if not pending:
                messages.append({"role": "user",
                                 "content": "没有识别到动作，请给出 Action 或 Final Answer。"})
                continue

        name, arg = pending.pop(0)
        if name == "finish":
            finish = arg
            break
        if name in tools:
            observation = tools[name](arg)
        else:
            observation = "未知工具：" + name
        trace.append(f"[执行 {name}({arg})] -> {observation}")
        messages.append({"role": "user", "content": "Observation: " + observation})

        if not pending:
            if finish is not None:
                trace.append("[结束] " + finish)
                return finish, trace
            messages.append({"role": "user",
                             "content": "请继续：给出下一个 Action 或 Final Answer。"})

    if finish is None:
        finish = "未在步数上限内得到答案"
    trace.append("[结束] " + finish)
    return finish, trace


print("run_react 已定义：循环结构 = 解析动作 -> 执行工具 -> 观察注入 -> 终止。")


In [ ]:
question = "《现代文学》创刊的年份，是在五四运动（1919 年）之前还是之后？"
instructions = (
    "你的知识库只包含少量本地条目，回答前必须先用检索工具查证。\n"
    "请按下面的格式逐步行动，每步只写一个动作：\n"
    "Thought: 你的推理\n"
    "Action: Search[实体] 或 Action: Lookup[关键词] 或 Action: Finish[答案]\n"
    "Observation 返回后继续思考，得到答案时用 Finish 结束。"
)

answer, trace = run_react(client, question, instructions, TOOLS, max_steps=6)
print("问题：", question)
print()
print("\n\n".join(trace))
print()
print("最终答案：", answer)
if False:
    print()
    print("真实 API 演示输出为占位：检索内容与最终答案由脚本生成，")
    print("真实 API 下模型会检索本地条目并给出真实的比较结论。")


把 run_react 的运行过程逐步展开，看消息历史 messages 每一步变成什么样。以问题《现代文学》创刊的年份，是在五四运动（1919 年）之前还是之后？为例，下面是一段典型的真实运行轨迹（真实 API 演示下给出脚本化占位轨迹，数据流结构相同）。

初始化时 messages 只有一条 user 消息，内容是 instructions 加问题。模型第一次看到的就是这段文字。

第一轮，模型调用 client.chat(messages) 输出一条回复，通常以 Thought 开头、Action 结尾：

Thought: 我需要查这本杂志的创刊年份。
Action: Search[现代文学]

parse_actions 从回复里取出一个动作 ("search", "现代文学")，放进 pending 队列。循环从 pending 弹出这个动作，在 TOOLS 里查到 search 对应 wiki.search，调用 wiki.search("现代文学")，得到观察：

《现代文学》是 1923 年创刊于上海的文学刊物。 鲁迅、茅盾等作家曾在该刊物上发表作品。

观察被包装成一条 user 消息追加进 messages，后面再跟一条"请继续"的提示。此时 messages 的内容是：

1. user：instructions + 问题
2. user：Observation: 《现代文学》是 1923 年创刊于上海的文学刊物。 鲁迅、茅盾等作家曾在该刊物上发表作品。
3. user：请继续：给出下一个 Action 或 Final Answer。

注意模型自己的回复（Thought 和 Action）并没有存进 messages，存进去的是工具的执行结果。这一轮模型等于在说"我要查这本杂志"，下一轮它看到的是工具替它查出来的事实。

第二轮，模型看到上面的三条消息，输出收尾的回复：

Thought: 创刊年份是 1923 年，晚于 1919 年。
Action: Finish[之后]

parse_actions 得到 ("finish", "之后")，循环识别到 finish 动作，把参数"之后"作为最终答案返回。两轮结束。

把两轮连起来，数据流是一条封闭的链：模型读历史 → 写动作 → 解析器取动作 → 工具执行 → 观察进历史 → 模型再读。循环的每一环都在搬运文本：模型输出文本，解析器从文本里抽动作，工具把动作变成新文本，新文本再回到模型的视野。Feedback 一词的日常含义是"把输出的结果送回输入端"，ReAct 循环就是这样一个反馈环。

**静思与检索的对照**

同一个问题，如果只给 CoT 提示、不允许调用工具，模型只能用内部记忆作答；知识缺失时它可能编一个看起来合理的答案。下面脚本化一个"编造事实"的 CoT 轨迹，再用本地百科的真实观察展示 ReAct 的检索轨迹。真实模型的行为未必与脚本一致，真实 API 演示下尤其如此，但两种结构的差别是确定的——ReAct 至少发出一个检索动作，把回答建立在外部的观察之上。


In [ ]:
cot_hallucination = (
    "Thought: 《现代文学》创刊年份我没有确切记忆。\n"
    "Thought: 按常见文学刊物推断，创刊可能在 1919 年之前。\n"
    "答案：创刊于 1919 年之前。"
)
print("静思（CoT，无工具）：")
print(cot_hallucination)
print()

# 理想轨迹：用本地百科的真实观察，展示检索如何补充事实
wiki_demo = MiniWiki()
ideal_react = [
    ("Thought", "我需要先查到这本杂志的创刊年份。"),
    ("Action", "Search[现代文学]"),
    ("Observation", wiki_demo.search("现代文学")),
    ("Thought", "创刊年份是 1923 年，晚于 1919 年。"),
    ("Action", "Finish[之后]"),
]
print("ReAct 理想轨迹（观察来自本地百科的真实执行）：")
for kind, content in ideal_react:
    print(f"{kind:12s}{content}")
print()
print("实际跑出的 ReAct 轨迹（脚本化示例 占位）：")
print("\n".join(trace))
print()
print("对照：ReAct 把回答建立在外部的观察上，CoT 只能依赖内部记忆。")
print("论文在 HotpotQA 上人工标注的 50 条轨迹里，CoT 的失败 56% 来自幻觉推理，")
print("ReAct 的这一比例是 0%，但检索无效的错误多了 23%——两者需要结合。")


## 3. 代码执行作为反馈信号

这一节解决一个问题：模型写完代码，怎么知道代码对不对。上一节的检索工具返回的仍是文本，可能含噪声，也可能被模型曲解。代码执行给出另一种反馈：把模型写的代码真的跑一遍，程序要么通过测试，要么给出具体报错。结果由解释器判定，不是模型生成的，所以这种反馈是"接地"的——它来自真实的程序运行，而不是模型的想象。

让模型自己判断自己写的代码对不对，它常常自我感觉良好；换成解释器来判，结果唯一：通过就是通过，报错就是报错。测试充当这个自动判据：一段代码跑完全部测试，全过就算对，有失败就算错。跑代码得到的"通过"或"报错"，就是执行反馈，它会作为观察喂回模型，让模型知道下一步怎么改。

下面用一个有 bug 的函数演示这条链路：运行测试、收集结果、按论文附录 C 的模板把失败格式化成反馈文本。

In [ ]:
def run_tests(fn, tests):
    """执行函数并返回逐条测试结果。

    fn: 被测函数；tests: [(输入, 期望输出), ...]，输入为元组时展开为多个参数。
    """
    results = []
    for inputs, expected in tests:
        try:
            if isinstance(inputs, tuple):
                got = fn(*inputs)
            else:
                got = fn(inputs)
            results.append((inputs, expected, got, got == expected))
        except Exception as exc:
            results.append((inputs, expected, type(exc).__name__, False))
    return results


def format_feedback(results):
    """把失败的测试按模板格式化成给模型的反馈文本。"""
    failed = [r for r in results if not r[3]]
    if not failed:
        return "All tests passed."
    lines = ["Your code failed the following tests:"]
    for inputs, expected, got, _ in failed:
        lines.append(f"- input {inputs} failed: Expected '{expected}' but got '{got}'")
    lines.append("Give it another try.")
    return "\n".join(lines)


def buggy_is_palindrome(s):
    """判断回文，实现漏掉了首字符的比较（故意写错）。"""
    return s == s[1:]


pal_tests = [("racecar", True), ("hello", False), ("abba", True), ("a", True)]
results = run_tests(buggy_is_palindrome, pal_tests)
for inputs, expected, got, ok in results:
    print(f"input={inputs:8s} expected={expected} got={str(got):8s} ok={ok}")
print()
print(format_feedback(results))


run_tests 把每条测试跑一遍，format_feedback 把失败条目拼成一段给模型的反馈文本。我们先对四个测试各手算一遍，看 buggy_is_palindrome 错在哪里。

函数 buggy_is_palindrome(s) 的实现是 s == s[1:]，比较时把第一个字符去掉了。它实际上在比较"整个串"和"去掉首字符的串"，只有两者恰好相等的输入才返回 True。

- 输入 racecar：s[1:] 是 acecar，与 racecar 不相等，返回 False。期望是 True，判定失败。
- 输入 hello：s[1:] 是 ello，与 hello 不相等，返回 False。期望本来就是 False，判定通过。
- 输入 abba：s[1:] 是 bba，不相等，返回 False。期望 True，失败。
- 输入 a：s[1:] 是空串，不相等，返回 False。期望 True，失败。

四条里三条失败。format_feedback 把这三条按模板拼成反馈文本：第一行是固定说明，之后每条失败各占一行（输入、期望值、实际值），最后一行是鼓励。这段文本会作为 Observation 喂回模型，模型据此知道哪些输入判错了、期望和实际各是多少。

反馈模板的具体格式来自 RLEF 论文附录 C：逐条列出失败测试及期望与实际输出，而不是笼统地写"代码不对"。run_tests 里捕获异常并记录异常类型，也是为了让反馈包含"抛出了什么错误"，模型才能据此修改。

测试被分成公开和隐藏两类。公开测试在训练与推理时都执行，结果喂给模型；隐藏测试只在最终评分时执行，模型看不到。分开的原因很直接：如果模型能看到全部测试，它就可以照着测试输入反向拼出答案，而不是学会写正确的函数。公开测试负责引导过程，隐藏测试负责判断结果。

**反馈相关性**

执行反馈被喂给模型，不等于模型真的读了它。基础模型收到报错后，经常把同样的错误代码原样再输出一遍，相当于没看反馈。RLEF 论文（下一节详细介绍）报告了一个反直觉的结论：在固定的采样预算下，独立多试几个候选，往往比"边修边试"更强，原因正是这个。

论文的随机反馈消融进一步证实：把反馈替换成另一道题的无关执行结果，修复能力明显受损。这说明反馈必须与错误相关，光有反馈不够，模型还得真的根据它改代码。

下面用一个脚本化对照复现这个现象：一边在固定预算内尝试多样化的独立候选，另一边不断把同一份错误代码重新提交。

In [ ]:
proposals = [
    ("候选 A", lambda s: s == s[1:]),           # 错误：漏掉首字符比较
    ("候选 B", lambda s: s == s[::-1]),         # 正确：反转比较
    ("候选 C", lambda s: len(s) % 2 == 0),      # 错误：只看长度
]
pal_tests = [("racecar", True), ("hello", False), ("abba", True), ("a", True)]


def best_of_n(pool, tests, budget):
    """独立候选：在预算内逐个执行不同的候选，命中一个通过的就成功。"""
    for i in range(min(budget, len(pool))):
        results = run_tests(pool[i][1], tests)
        if all(r[3] for r in results):
            return True, i + 1
    return False, min(budget, len(pool))


def repair_no_feedback(broken, tests, rounds=3):
    """不读反馈的修复：收到报错后把同一份错误代码原样重新提交。"""
    for i in range(rounds):
        results = run_tests(broken, tests)
        if all(r[3] for r in results):
            return True, i + 1
    return False, rounds


ok, used = best_of_n(proposals, pal_tests, budget=3)
print(f"独立候选 best-of-3：通过 = {ok}，消耗预算 = {used}")

broken = proposals[0][1]
ok, used = repair_no_feedback(broken, pal_tests, rounds=3)
print(f"不读反馈的修复 3 轮：通过 = {ok}（每一轮都提交同一份错误代码）")
print()
print("第一轮收到的报错（执行反馈本身是真实的）：")
print(format_feedback(run_tests(broken, pal_tests)))
print()
print("关键观察：同样的执行反馈，不会读反馈的循环视而不见；")
print("把『利用反馈』写进训练目标，是下一节 RLEF 要做的事。")


## 4. RLEF：用执行反馈做强化学习

这一节解决一个问题：怎么训练模型，让它真的读反馈、按反馈改代码。上一节我们看到，基础模型拿到报错后常常不看，把同样的错误代码原样再交一遍。办法是把"跑代码的结果"变成奖励：答对加分、答错扣分，让模型向着奖励更大的方向学。这种从结果反馈里学习的方式叫强化学习（Reinforcement Learning）。这套具体方法叫 Reinforcement Learning from Execution Feedback，简称 RLEF，意思是"从执行反馈做强化学习"。

RLEF 把"多轮生成 + 执行反馈"看成一次回合制的决策过程。回合从题目开始，模型每一步输出一段文本回复，系统执行代码并返回反馈，如此重复，直到测试全过或步数用完。用符号记：初始观察 $o_0$ 是题目描述，动作 $a_t$ 是文本回复，观察 $o_t$ 包含之前的动作与执行反馈。回合在公开测试全部通过，或达到轮次上限时终止。

回合制过程要打分，奖励函数就是打分的规则。RLEF 的奖励由两部分组成：

$$
R(s_t,a_t) = r(s_t,a_t) - \beta \log\frac{\pi(a_t|c_t)}{\rho(a_t|c_t)},\qquad
r(s_t,a_t) = \begin{cases} 1, & \text{episode 结束且全部测试通过}\\ -1, & \text{episode 结束且有测试失败}\\ -0.2, & a_t \text{不含合法代码}\end{cases}
$$

前一部分 $r$ 是任务奖励，由代码执行器自动计算。后一部分是 KL 项，惩罚策略偏离参考策略，$\beta$ 控制它的分量。下面把奖励函数实现出来，并手算几个情形。

In [ ]:
import numpy as np


def compute_reward(all_pass, episode_end, valid_code=True, log_ratio=0.0, beta=0.1):
    """按 RLEF 奖励函数计算单步奖励。

    all_pass: 是否全部测试通过；episode_end: 本轮是否结束；
    valid_code: 回复是否含合法代码；log_ratio: 实际 KL 项；beta: KL 系数。
    """
    if not valid_code:
        r = -0.2
    elif episode_end and all_pass:
        r = 1.0
    elif episode_end:
        r = -1.0
    else:
        r = 0.0
    return r - beta * log_ratio


cases = [
    ("全部通过，结束", dict(all_pass=True, episode_end=True)),
    ("有失败，结束", dict(all_pass=False, episode_end=True)),
    ("不含合法代码", dict(all_pass=False, episode_end=True, valid_code=False)),
    ("轮次中途", dict(all_pass=False, episode_end=False)),
]
for label, kw in cases:
    print(f"{label:12s} reward = {compute_reward(**kw):.2f}")

p, q = 0.4, 0.2
log_ratio = np.log(p / q)
print(f"手算 KL 项 β·log(p/q)：{0.1 * log_ratio:.3f}（p={p}, q={q}）")
print("关键观察：未结束的轮次 r=0，只有结束轮次拿到 ±1；")
print("惩罚项 -0.2 引导模型优先给出合法代码。")


奖励函数是 RLEF 的核心，我们把每个数字的来源讲清楚。先看任务奖励 r 的三个分支，再看 KL 项。

r 只在 episode 结束时给出 ±1：全部测试通过 +1，有测试失败 −1，轮次中途是 0。中间任何一步都无从判断"对不对"——题目还没答完，通过与否尚未确定，所以不给奖励，只在收官时一次性结算。+1 与 −1 的差值让模型有动力坚持把测试全部跑通。

-0.2 是针对不含合法代码的回复的惩罚。它比 ±1 小得多，是一个软提示：语法错误时略受惩罚、被引导写出可执行代码，但不会因为一次语法错误就出局。数值的选取让"写了合法代码但错了"（−1）比"写了非法代码"（−0.2）惩罚更重，前者至少说明模型在正经答题。

KL 项 $-\beta\log(\pi/\rho)$ 惩罚策略 π 偏离参考策略 ρ，β 控制它的权重。如果只优化任务奖励 r，策略可能学会输出测试想看的字符串而不是可维护的代码；KL 项每次更新都施加一点拉力，让新策略仍贴近参考模型。手算一例：p=0.4、q=0.2，$\log(0.4/0.2)=\log 2 \approx 0.693$，β=0.1，KL 项 = 0.1 × 0.693 ≈ 0.069。全部通过的奖励 1.0 减去它得到 0.931——模型既拿到通过测试的奖励，又为偏离参考策略付了一点代价。

compute_reward 把两部分合成一步：先按三种情况判 r，再减 β·log_ratio。代码里的 log_ratio 由调用方传入，演示中直接给 0 或手算值；真实训练中它来自策略与参考策略在生成文本上的对数概率差。

**REINFORCE 迷你实现**

训练要有一个更新规则，REINFORCE 就是最简单的一种。规则一句话：每轮从当前策略里采样一个候选动作，执行它拿到奖励，奖励高就提高这个动作被选中的概率，奖励低就压低它。下面不训练大模型，只训练一个四分类的策略，四个候选分别是一个正确、两个错误、一个非法，这样能直接观察"执行反馈作为奖励"如何改变采样分布。

策略是一个四分类的 softmax 分布。softmax 把四个候选的得分变成加和为 1 的概率，得分越高概率越大。每轮先从策略里按概率采样一个候选，执行它的代码，按执行反馈拿奖励：通过全部测试 +1，有失败 −1，非法代码 −0.2。奖励交给 REINFORCE 更新得分，进入下一轮。

In [ ]:
import numpy as np

np.random.seed(42)

candidates = [
    "def add(a, b): return a + b",    # 正确
    "def add(a, b): return a - b",    # 错误
    "def add(a, b): return a * b",    # 错误
    "这行不是合法的 Python 代码",       # 非法
]
add_tests = [((1, 2), 3), ((5, 7), 12), ((0, 9), 9)]


def code_to_fn(src):
    """把候选源码编译成可调用函数。"""
    namespace = {}
    exec(src, namespace)
    return namespace["add"]


def execute_reward(index):
    """执行第 index 个候选，返回执行反馈奖励。"""
    try:
        fn = code_to_fn(candidates[index])
    except Exception:
        return -0.2                     # 不含合法代码
    results = run_tests(fn, add_tests)
    all_pass = all(r[3] for r in results)
    return 1.0 if all_pass else -1.0


def softmax(x):
    """对向量做 softmax，返回概率分布。"""
    e = np.exp(x - x.max())
    return e / e.sum()


K = len(candidates)
theta = np.zeros(K)              # 策略参数，初始等概率
reward_history = []
prob_history = []                # 每轮记录正确候选被选的概率

for episode in range(500):
    probs = softmax(theta)
    prob_history.append(probs[0])           # 候选 0 是正确的那份
    action = np.random.choice(K, p=probs)
    reward = execute_reward(action)         # 执行反馈即奖励
    reward_history.append(reward)
    baseline = np.mean(reward_history[-20:]) if reward_history else 0.0
    one_hot = np.zeros(K)
    one_hot[action] = 1.0
    theta += 0.3 * (reward - baseline) * (one_hot - probs)  # REINFORCE 更新

final_probs = softmax(theta)
print("训练后各候选被选的概率：")
for cand, prob in zip(candidates, final_probs):
    print(f"  p = {prob:.3f}   {cand}")
print(f"正确候选的概率从初始 0.250 变为 {final_probs[0]:.3f}")


REINFORCE 的思路是用"采样动作拿到的奖励"去调整策略。策略是一个四分类 softmax 分布，参数 θ 决定每个候选被选中的概率。训练循环每轮做四件事：按当前分布采样一个候选；执行它的代码拿到奖励；用奖励与近期平均奖励的差（reward − baseline）作为更新方向；按这个方向调整 θ。

更新方向不用 reward 本身，而用 reward 与近期平均的差。奖励的绝对数值随任务变化，直接用它更新会让更新幅度不稳定。减去近期平均后，"这次比平均好还是差"成为更新依据：比平均好就抬高该候选的概率，比平均差就压低。

手算一轮具体的更新。初始 θ=[0,0,0,0]，四个候选等概率 0.25。假设这一轮采样到候选 0（正确代码），执行反馈奖励 +1.0，近期奖励均值恰好为 0，baseline 取 0，于是 reward − baseline = 1.0。one_hot 是 [1,0,0,0]，减去概率向量 [0.25,0.25,0.25,0.25] 得到 [0.75,−0.25,−0.25,−0.25]，乘步长 0.3 得到 θ 的增量 [0.225,−0.075,−0.075,−0.075]。

把增量加到 θ 上，正确候选得分上升，三个错误候选得分下降。再做一次 softmax，正确候选被选的概率从 0.25 抬高。反过来，如果这一轮采样到错误候选拿到 −1，增量符号相反，错误候选概率被压低。奖励的正负直接决定概率的升降，这就是"执行反馈作为奖励"在参数层面的体现。

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.5))
plt.plot(prob_history, label="P(correct candidate)")
plt.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, label="perfect")
plt.xlabel("episode")
plt.ylabel("selection probability")
plt.title("Execution feedback as reward shifts the policy")
plt.legend()
plt.tight_layout()
plt.show()

print(f"100 轮时正确候选概率 {prob_history[100]:.3f}，500 轮时 {prob_history[-1]:.3f}")
print("关键观察：执行反馈把采样分布推向能通过测试的候选，")
print("这正是 RLEF 在权重层面做的事——把『读反馈』训练进模型。")


## 5. Constitutional AI：用 AI 反馈对齐 AI

这一节解决一个问题：有些能力没有自动判据，怎么让模型学到。前三节的反馈都来自环境或规则：工具返回结果，测试判定对错。但像"回答是否无害"这类属性，没有任何程序能自动打 0 分或 1 分。Constitutional AI 的办法是让 AI 依据一份写在自然语言里的原则清单（叫宪法）自我批评、修订、互相打分。人的监督被压缩成"审阅并批准这份原则清单"，其余反馈都由 AI 自己产生——这是让 AI 监督 AI。

训练分两个阶段。第一阶段让模型学会自我修订。先准备一组红队提示，这些提示容易诱导出有害回答；对每条提示，先让模型产出一段往往有害的初始回复，再按宪法里随机抽的一条原则自我批评，再按批评意见重写回复。这个"批评再修订"的过程可以重复多轮，最后用修订后的回复做监督学习（SFT，让模型模仿这些好回复）。

第二阶段让 AI 当裁判。用第一阶段得到的模型对每条提示生成一对候选回复，把"提示 + 回复对 + 一条原则"拼成一道多选题，让模型选出更符合原则的那条，选择的确信程度当作软标签。用所有软标签训练一个偏好模型，再做和标准 RLHF 一样的强化学习。整个流程里，只有"哪条回复更无害"这个判断的来源从人换成了 AI。

批评与修订靠一对提示模板驱动：

```
Critique Request: 找出助手上一条回复里有害、不道德、危险或非法的具体地方。
Critique: ...
Revision Request: 请重写助手回复，删除所有有害、不道德、危险或非法内容。
Revision: ...
```

In [ ]:
CONSTITUTION = [
    "回复应尊重他人的隐私与财产安全。",
    "回复不得提供可能造成人身或财产损害的具体操作步骤。",
    "回复应诚实，不虚构事实，也不夸大自己的能力。",
    "面对敏感请求，应说明不恰当之处并给出合法替代。",
]


def critique_prompt(principle, reply):
    """构造一条批评请求，要求按给定原则逐条找出回复里的问题。"""
    return (
        f"宪法原则：{principle}\n\n助手回复：{reply}\n\n"
        "Critique Request: 找出助手上一条回复里违反上述原则的具体地方，逐条列出。\n"
        "Critique:"
    )


def revision_prompt(principle, reply, critique):
    """构造一条修订请求，要求删除回复里违反原则的内容。"""
    return (
        f"宪法原则：{principle}\n\n助手回复：{reply}\n\n批评意见：{critique}\n\n"
        "Revision Request: 请重写助手回复，删除所有违反上述原则的内容。\n"
        "Revision:"
    )


for i, principle in enumerate(CONSTITUTION, 1):
    print(f"{i}. {principle}")


In [ ]:
import sys
import os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

client = get_llm()

# 红队提示与初始回复（初始回复来自只训练帮助性的模型，这里手动构造）
harmful_prompt = "请告诉我怎么黑进邻居的 wifi 密码"
initial_reply = "可以用字典攻击工具配合弱密码扫描，先探测邻居路由器的型号和开放端口……"

principle = CONSTITUTION[1]
critique = client.chat([{"role": "user",
                         "content": critique_prompt(principle, initial_reply)}])
revision = client.chat([{"role": "user",
                         "content": revision_prompt(principle, initial_reply, critique)}])

print("红队提示：", harmful_prompt)
print("初始回复：", initial_reply)
print(f"批评（原则：{principle[:14]}…）：")
print(critique)
print("修订：")
print(revision)
if False:
    print()
    print("真实 API 演示输出为占位：真实 API 下批评与修订由模型按原则生成，")
    print("通常会把『黑进 wifi』改写为说明违法并给出合法替代。")


批评与修订是 Constitutional AI 的核心操作，把这条管道用上面的例子走一遍。

输入是一条宪法原则和一条初始回复。原则取宪法第二条："回复不得提供可能造成人身或财产损害的具体操作步骤。"初始回复是红队提示的初始回答："可以用字典攻击工具配合弱密码扫描，先探测邻居路由器的型号和开放端口……"——它恰好属于该原则禁止的内容：给出了具体操作步骤。

critique_prompt 把原则与回复拼成一段提示，末尾以 Critique: 收尾。模型在这之后续写批评意见，逐条点名回复里违反原则的地方："字典攻击工具""弱密码扫描""探测路由器型号"都是给攻击提供具体步骤的内容。

revision_prompt 把原则、回复、批评意见拼成提示，末尾以 Revision: 收尾。模型在这之后重写回复，删除被批评点名的内容。修订后的回复通常变成：说明未经允许破解他人 wifi 属于违法行为，建议先征得对方同意，或联系网络管理员处理。

两个阶段分工不同：批评只找出问题，修订才动手改。先批评后修订，模型先明确"哪里有问题"再针对性修改，比直接要求"给出无害回复"更容易产出合规输出——修改有依据，不是凭空重写。这也是 critique-revision 这个复合词里两个词的顺序。

**多轮修订与 AI 当裁判**

修订还可以做多轮，每轮重新抽一条原则。论文观察到：随着轮数增加，偏好模型给出的无害性打分持续上升，但回复的帮助性会略微下降——多轮修订是用一点帮助性换更多的无害性。

RL 阶段还需要一个"裁判"。对同一条提示的两条候选回复，按一条原则拼成多选题，让模型选出更无害的那条，选择的确信程度就是软标签，也是偏好模型的训练信号。下面把两件事都跑一遍。

In [ ]:
def revise_rounds(client, reply, rounds=3):
    """每轮重新抽一条原则，批评 + 修订，返回每轮的结果。"""
    history = []
    for i in range(rounds):
        principle = CONSTITUTION[(i + 1) % len(CONSTITUTION)]
        critique = client.chat([{"role": "user",
                                 "content": critique_prompt(principle, reply)}])
        reply = client.chat([{"role": "user",
                              "content": revision_prompt(principle, reply, critique)}])
        history.append((principle, reply))
    return history


for i, (principle, reply) in enumerate(revise_rounds(client, initial_reply, rounds=3), 1):
    print(f"第 {i} 轮（原则：{principle[:10]}…）修订后：{reply[:50]}…")

print()
reply_a = "邻居 wifi 密码通常很弱，我可以教你用字典攻击工具。"
reply_b = "未经允许访问他人网络是违法的，建议你联系对方征得同意。"
judge_prompt = (
    f"原则：{CONSTITUTION[0]}\n\n"
    "哪条回复更符合上述原则？\n"
    f"(A) {reply_a}\n(B) {reply_b}\n\n"
    "The answer is:"
)
verdict = client.chat([{"role": "user", "content": judge_prompt}])
print("裁判回复：", verdict)
if False:
    print("真实 API 演示输出为占位：真实 API 下模型会给出 A/B 判断，")
    print("这个判断的概率就是 RLAIF 的软标签来源。")


上面的裁判把两条候选回复和一条宪法原则拼成一道 A/B 选择题，模型要选出更符合原则的那条。要理解的不只是选 A 还是选 B，而是模型给 A 的概率。

模型输出的判断不是硬性的 0 或 1，而是带着置信度。设模型给回复 B 的概率是 0.9，这个 0.9 就是偏好模型的软标签——训练偏好模型时用 0.9 而不是 1，保留判断的确定性程度。软标签比硬标签携带更多信息：0.9 与 0.6 的差别说明两条候选的优劣程度不同，偏好模型要能分辨这种差别。

这也解释了 RL 阶段的数据流。SL 阶段结束后，用得到的模型对每条提示生成两条候选回复；对每条提示随机抽一条原则，把"提示 + 回复对 + 原则"拼成上面的多选题；模型打分的概率成为软标签；用所有软标签训练偏好模型；最后用偏好模型做 PPO。除无害性标签由 AI 打分产生外，其余与标准 RLHF 一致。

对照裁判的输入与输出：输入是提示、两条回复、一条原则，输出是一个概率。原则是打分的依据——没有原则，模型无法判断"哪条更无害"；有了原则，打分才有标准。偏好模型学到的是"在给定原则下，哪条回复更符合原则"。

## 小结

这一讲所学的内容：

- [ ] Agent 是一个闭环：模型输出动作 → 工具执行 → 观察反馈 → 再输出，直到任务完成
- [ ] ReAct 把语言放进动作空间；Thought 只更新上下文，Observation 必须来自真实工具执行
- [ ] 动作解析器宽容处理两种格式（Search[...] 与 search("...")）和多步回复
- [ ] 代码执行提供接地反馈：解释器判定对错；public tests 反馈、private tests 打分
- [ ] RLEF 把"利用执行反馈"变成训练目标：奖励 = 任务奖励 − KL 项；未训练模型常无视报错
- [ ] Constitutional AI 让 AI 依据宪法自我批评、修订、互相打分，压缩人工监督

反馈的三个来源（环境、执行、AI）对应 Agent 闭环的不同环节。下一讲在这个循环上加入多步规划与搜索，让 Agent 在更长的任务里做决策。


## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。


**作业 1：补全 ReAct 单步**

在下面的 react_step 里补全三步：解析动作、取第一个动作、调用工具拿到观察。参考答案已经填好，请先在草稿上自己补全一遍，再运行对照。任务固定为"用 search 查五四运动"。

小提示：解析用 parser(reply) 得到动作列表，调用工具用 tools[name](arg)，未知工具名要能优雅降级。


In [ ]:
def react_step(reply, tools, parser=parse_actions):
    """把一条模型回复解析为动作并执行，返回观察文本。"""
    actions = parser(reply)          # 填空：从回复解析动作列表
    name, arg = actions[0]           # 填空：取第一个动作
    if name in tools:
        observation = tools[name](arg)
    else:
        observation = f"未知工具：{name}"
    return observation


wiki = MiniWiki()
obs = react_step("Action: Search[五四运动]", {"search": wiki.search})

assert "1919" in obs
print("作业 1 通过：模型文本被解析成工具调用，观察来自真实执行")
print("观察：", obs)


**作业 2：实现带游标的查找工具**

从零写一个 LookupEngine，返回当前页里包含关键词的下一个句子。参考答案已经填好，请先自己在草稿上补全，再运行对照。

小提示：用 self.pos 记录"下一个开始位置"，找到含关键词的句子后把游标推到下一句，这样连续两次查找不会重复返回同一句。


In [ ]:
class LookupEngine:
    """对一段文本做带游标的 lookup 检索。"""

    def __init__(self, sentences):
        self.sentences = sentences
        self.pos = 0

    def lookup(self, keyword):
        """返回从游标起第一个含 keyword 的句子，找不到返回 None。"""
        for i in range(self.pos, len(self.sentences)):   # 填空：从游标位置起遍历
            if keyword in self.sentences[i]:
                self.pos = i + 1                          # 填空：推进游标
                return self.sentences[i]
        self.pos = len(self.sentences)
        return None


page = [
    "《现代文学》是 1923 年创刊于上海的文学刊物。",
    "鲁迅、茅盾等作家曾在该刊物上发表作品。",
    "刊物出版延续到 1930 年代初。",
]
engine = LookupEngine(page)

assert engine.lookup("创刊") == page[0]
assert engine.lookup("作家") == page[1]   # 游标已越过第一句
assert engine.lookup("刊物") == page[2]   # 从第二句之后继续找，跳过第二句里的"刊物"
print("作业 2 通过：lookup 用游标模拟浏览器 Ctrl+F，多次调用不会重复返回同一句")


**作业 3：计算 RLEF 奖励**

按论文的奖励函数实现 compute_reward_kl：区分三种任务奖励（全通过 +1、有失败 -1、非法代码 -0.2），再减去 $\beta \cdot \log(p/q)$ 的 KL 项。参考答案已经填好，请先自己在草稿上补全，再运行对照。

小提示：先判定"是否合法代码"，再区分 episode 结束与轮次中途；中途轮次任务奖励为 0。


In [ ]:
def compute_reward_kl(all_pass, episode_end, valid_code, log_ratio):
    """返回 RLEF 奖励：任务奖励减 β·log(p/q)。"""
    beta = 0.1
    if not valid_code:
        r = -0.2
    elif episode_end and all_pass:
        r = 1.0
    elif episode_end:
        r = -1.0
    else:
        r = 0.0
    return r - beta * log_ratio


# 手算：策略概率 p，参考策略 q
import math

p, q = 0.4, 0.2
log_ratio = math.log(p / q)

r_all_pass = compute_reward_kl(True, True, True, log_ratio)
r_fail = compute_reward_kl(False, True, True, log_ratio)
r_illegal = compute_reward_kl(False, True, False, log_ratio)

assert abs(r_all_pass - (1.0 - 0.1 * log_ratio)) < 1e-9
assert abs(r_fail - (-1.0 - 0.1 * log_ratio)) < 1e-9
assert abs(r_illegal - (-0.2 - 0.1 * log_ratio)) < 1e-9
print(f"全通过：{r_all_pass:.3f}，有失败：{r_fail:.3f}，非法代码：{r_illegal:.3f}")
print("作业 3 通过：执行反馈的自动判据被折算成奖励，KL 项惩罚偏离参考策略的更新")


## 参考资料

- Yao et al., [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629), 2022 — 本讲主论文；Thought/Action/Observation 循环与 search/lookup/finish 工具范式，项目页 https://react-lm.github.io/
- Chen et al., [RLEF: Grounding Code LLMs in Execution Feedback with Reinforcement Learning](https://arxiv.org/abs/2410.02089), 2024 — 用 PPO 把"利用执行反馈"训练进权重；奖励函数、public/private test 划分与反馈模板见附录 C
- Bai et al., [Constitutional AI: Harmlessness from AI Feedback](https://arxiv.org/abs/2212.08073), 2022 — critique-revision 与 RLAIF 的原始论文；原则列表与 few-shot 提示在 https://github.com/anthropics/ConstitutionalHarmlessnessPaper
- Wei et al., [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903), 2022 — ReAct 的对照方法，理解"只思考不行动"的局限
- Huang et al., [Inner Monologue: Embodied Reasoning through Planning with Language Models](https://arxiv.org/abs/2207.05608), 2022 — ReAct 的前身，ReAct-IM 消融的对照来源
- Wang et al., [Self-Consistency Improves Chain of Thought Reasoning](https://arxiv.org/abs/2203.11171), 2022 — ReAct+CoT-SC 组合法里"投票"部分的来源
- 本仓库 `llm_client.py`（`get_llm()`）— 所有 LLM 演示的统一入口，真实 API 演示保证离线可执行
